# POC Extraction de données

Extraction des données provenant des rapport de G2K vers un format plus simple.

Import, définition des variables et récupération du contenue du rapport (sous forme de `string`)

In [3]:
import pandas as pd
import numpy as np
import re

## Variable global
content = ""
sections_content = {}
sections_titles = []

# Récuperaction du contenue du rapport
with open("../data/rapport-RGU_3cm.txt", "r") as f:
    content = f.read()

Définition de mes *regex*

In [4]:
section_header_pattern          = re.compile(r"^\*+$\n^\*{5}(.*)\*{5}$\n^\*+$", re.MULTILINE)
s1_kv_pattern                   = re.compile(r"^([A-ZÀ-Ÿ][a-zA-ZÀ-ÿ' ]+?)\s*:\s*(.*)$", re.MULTILINE)
s2_header_pattern               = re.compile(r"(Numéro)\s+(Début)\s+-\s+(Fin)\s+(Centroïde)\s+(Energie)\s+(FWHM)\s+(Surface)\s+(Incert\.)\s+(Fond sous)\s*\r?\n^\s*(du pic)\s+(\(canaux\))\s+(\(keV\))\s+(\(keV\))\s+(le pic)", re.MULTILINE)
s2_data_pattern                 = re.compile(r"^\s*([MmF]?)\s*(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+-\s*(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)", re.MULTILINE)
s3_header_pattern               = re.compile(r"^\s+(Nom\sdu)\s+(Indice\sde)\s+(Energie)\s+(Intensité)\s+(Activité)\s+(Incert\.)$\n^\s+(nucléide)\s+(confiance)\s+(\W\w+\W)\s+(\W%\W)\s+(\WmBq\Wg\s+\W)\s+(\WmBq\Wg\s+\W)\n", re.MULTILINE)
s3_data_pattern                 = re.compile(r"^\s*([A-Z]{1,2}-\d{1,3})?\s*(\d+\.\d+)?\s*(\d+\.\d+)(\*?)\s*(@?)\s*(\d+\.\d+)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)$", re.MULTILINE)
s4_header_1_pattern             = re.compile(r"^\s*(Nom du)\s+(Indice de)\s+(Activité moyenne)\s+(Incert\.)$\n^\s+(nucléide)\s+(confiance)\s+(pondérée)$\n^\s+(\WmBq\Wg\s+\W)\s+(\WmBq\Wg\s+\W)$", re.MULTILINE)
s4_header_2_pattern             = re.compile(r"^\s*(Numéro)\s+(Energie)\s+(Intensité)\s+(Incert\.)\s+(Type)\s+(Nucléide)$\n^\s+(du pic)\s+(\WkeV\W)\s+(\Wcoups\Wsec\W)\s+(du pic)\s+(potentiel)$", re.MULTILINE)
s4_data_1_pattern               = re.compile(r"^\s*(X?)\s+([A-Z]{1,2}-\d{1,3})\s*(@?)\s+(\d+\.\d+)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)?\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)?", re.MULTILINE)
s4_data_2_pattern               = re.compile(r"^\s+([MmF])?\s*(\d+)\s+(\d+\.\d+)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+\.\d+)\s*(Sum|D-Esc\.|S-Esc\.|Tol\.)?\s*([A-Z]{1,2}-\d{1,3})?\s*$", re.MULTILINE)
s5_header_pattern               = re.compile(r"^\s+(Nom\sdu)\s+(Energie)\s+(Intensité)\s*(LD\sEnergie)\s*(LD\snucléide)\s+(Activité)\s+(SD\sEnergie)$\n^\s+(nucléide)\s+(\WkeV\W)\s+(\W%\W)\s+(\WmBq\Wg\s+\W)\s+(\WmBq\Wg\s+\W)\s+(\WmBq\Wg\s+\W)\s+\s+(\WmBq\Wg\s+\W)$", re.MULTILINE)
s5_data_pattern                 = re.compile(r"^\s*(\+?)\s*(\??)\s*(>?)\s*([A-Z]{1,2}-\d{1,3})?\s+(\d+\.\d+)(\*?)\s*(@?)\s+(\d+\.\d+)\s+([+\-]?\d+(?:\.\d+)?(?:E[+\-]?\d+)?|Non\sCalc)(?:\s*([+\-]?\d+(?:\.\d+)?(?:E[+\-]?\d+)?))?\s+([+\-]?\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+([+\-]?\d+(?:\.\d+)?(?:E[+\-]?\d+)?)$", re.MULTILINE)
s6_header_pattern               = re.compile(r"^\s+(.*)$\n^\s+(.*)$", re.MULTILINE)
s6_word_column_pattern          = re.compile(r"([A-Za-zÀ-ÿ]+\.?)")
s6_nucleide_line_pattern        = re.compile(r"^[+>?]\s+([A-Z]{1,2}-\d{1,3})\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)", re.MULTILINE)


## Extraction des données de chaque section

### Notes
**TODO**
- Pour les sections de 2 à 6 extraire les métadonnées.
- Faire un `ffill` pour les noms de nucléide

### Setup
1 - On extrait les titre et on créer un dictionnaire de DataFrame

2- On découpe `content` par section que l'on range dans un dictionnaire `sections_content`

In [5]:
section_headers = section_header_pattern.findall(content)

# 1
for i, section in enumerate(section_headers):
    sections_titles.append(section.strip())

# 2
matches = list(section_header_pattern.finditer(content))
for i, match in enumerate(matches):
    title = match.group(1).strip()
    start = match.end()
    end = matches[i + 1].start() if i + 1 < len(matches) else len(content)
    sections_content[title] = content[start:end].strip()

### S1 - RAPPORT DE L’ANALYSE DU SPECTRE
C'est les metadata du rapport.

Pour l'instant ce n'est pas encore bon:
Mauvaise dections des clés/valeurs

In [6]:
content_s1 = sections_content[sections_titles[0]]
matches = s1_kv_pattern.findall(content_s1)
data_s1 = {key.strip(): value.strip() for key, value in matches}

####### DATA ##########
data_s1

{'Nom du fichier': 'C:\\GENIE2K\\CAMFILES\\SPECTRES\\STANDARDS\\Standard_Syrah_2024\\S',
 'Date du rapport': '02/06/2026 11:08:46',
 'Description': 'ech',
 'Identification': "Type d'échantillon          :",
 'Géométrie de mesure': 'Sensibilité de recherche    :  2.50',
 'Zone de recherche des pics': '50 - 16384',
 'Zone de calcul des surfaces': '50 - 16384',
 'Tolérance sur les énergies': '1.000 keV',
 'Date de mesure': '23/02/2024 11:38:52',
 'Temps actif': '255962.3 secondes',
 'Temps réel': '256073.1 secondes',
 'Temps mort': '0.04 %',
 'Nom de la géométrie': 'Rapport sur Les paramètres des pics  02/06/2026 11:08:46          Page  2'}

### S2 - RAPPORT ANALYSE DES PICS

In [7]:
def extract_header_s2(content):
    match = re.search(s2_header_pattern, content)
  
    if not match:
        return None
        
    columns = [
        "Marker",                                   # Marqueur (M, m, F)
        f"{match.group(1)} {match.group(9)}",       # Numéro du pic
        f"{match.group(2)} {match.group(10)}",        # Début (canaux)       
        f"{match.group(3)} {match.group(11)}",                       # Fin (canaux)
        match.group(4),                             # Centroïde
        f"{match.group(5)} {match.group(12)}",      # Energie (keV)
        f"{match.group(6)} {match.group(13)}",      # FWHM (keV)
        match.group(7),                             # Surface
        match.group(8),                             # Incert.
        f"{match.group(8)} {match.group(13)}"       # Fond sous le pic
    ]
    
    return columns



In [ ]:
def extract_data_s2(content, header):
    matches = re.findall(s2_data_pattern, content)
    df = pd.DataFrame(matches, columns=header)

    # Replace all instances of empty string with NaN in column: 'Marker'
    df['Marker'] = df['Marker'].replace('', np.nan)

    # Change column type to category for column: 'Marker'
    df = df.astype({'Marker': 'category'})

    # Change column type to float64 for columns: 'Numéro Fond sous', 'Début du pic' and 7 other columns
    df = df.astype({'Numéro Fond sous': 'float64', 'Début du pic': 'float64', 'Fin (canaux)': 'float64', 'Centroïde': 'float64', 'Energie (keV)': 'float64', 'FWHM (keV)': 'float64', 'Surface': 'float64', 'Incert.': 'float64', 'Incert. (keV)': 'float64'})

    return df


In [9]:
content_s2 = sections_content[sections_titles[1]]
header_s2 = extract_header_s2(content_s2)
data_s2 = extract_data_s2(content_s2, header_s2)

####### DATA ##########
data_s2

,Marker,Numéro Fond sous,Début du pic,Fin (canaux),Centroïde,Energie (keV),FWHM (keV),Surface,Incert.,Incert. (keV)
0,,1,80,97,89.94,16.21,1.27,9.406E+03,650.88,2.958E+04
1,,2,98,114,106.38,19.26,1.43,4.954E+03,612.65,2.841E+04
2,M,3,130,158,141.01,25.67,0.85,4.291E+03,269.06,2.308E+04
3,m,4,130,158,150.25,27.38,0.86,3.786E+03,266.29,2.492E+04
4,,5,191,205,200.49,36.68,0.50,5.911E+02,525.79,2.378E+04
...,...,...,...,...,...,...,...,...,...,...
181,,182,15550,15569,15559.19,2879.40,0.28,2.317E+01,15.24,9.833E+00
182,,183,15777,15796,15786.26,2921.43,1.21,2.569E+01,18.45,1.631E+01
183,,184,15872,15891,15881.97,2939.14,0.35,1.900E+01,12.08,5.000E+00
184,,185,16081,16102,16091.77,2977.97,0.84,2.492E+01,13.29,5.083E+00


### S3 - RAPPORT IDENTIFICATION DES NUCLEIDES

In [10]:
def extract_header_s3(content):
    matches = re.search(s3_header_pattern, content)

    if not matches:
        return None
    
    columns = [
        f"{matches.group(1)} {matches.group(7)}",
        f"{matches.group(2)} {matches.group(8)}",
        f"{matches.group(3)} {matches.group(9)}",
        "Marker *",
        "Marker @",
        f"{matches.group(4)} {matches.group(10)}",
        f"{matches.group(5)} {matches.group(11)}",
        f"{matches.group(6)} {matches.group(12)}"
    ]
   
    return columns

In [11]:
def extract_data_s3(content, header):
    matches = re.findall(s3_data_pattern, content)
    df = pd.DataFrame(matches ,columns=header)
    
    # Loaded variable 'data_s3' from kernel state

    # Replace all instances of empty string with Nan in columns: 'Nom du nucléide', 'Indice de confiance'
    df['Nom du nucléide'] = df['Nom du nucléide'].replace('', np.nan)
    df['Indice de confiance'] = df['Indice de confiance'].replace('', np.nan)

    # Change column type to float64 for columns: 'Energie (keV)', 'Intensité (%)' and 3 other columns
    df = df.astype({'Energie (keV)': 'float64', 'Intensité (%)': 'float64', 'Activité (mBq/g   )': 'float64', 'Incert. (mBq/g   )': 'float64', 'Indice de confiance': 'float64'})

    # Replace gaps forward from the previous valid value in: 'Nom du nucléide'
    df = df.fillna({'Nom du nucléide': df['Nom du nucléide'].ffill()})

    # Change column type to category for column: 'Nom du nucléide'
    df = df.astype({'Nom du nucléide': 'category'})

    # Change column type to bool for columns: 'Marker *', 'Marker @'
    df = df.astype({'Marker *': 'bool', 'Marker @': 'bool'})
    
    return df

In [12]:
content_s3 = sections_content[sections_titles[2]]
header_s3 = extract_header_s3(content_s3)
data_s3 = extract_data_s3(content_s3, header_s3)

data_s3

,Nom du nucléide,Indice de confiance,Energie (keV),Marker *,Marker @,Intensité (%),Activité (mBq/g ),Incert. (mBq/g )
0,PB-210,1.000,46.53,True,False,4.25,4956.55800,83.101250
1,BI-214,1.000,609.31,True,False,46.30,4851.70700,60.213090
2,BI-214,NaN,768.36,True,True,5.04,2096.21800,104.498600
3,BI-214,NaN,806.17,True,True,1.23,1789.53500,289.360100
4,BI-214,NaN,934.06,True,True,3.21,2163.14500,156.012700
5,BI-214,NaN,1120.29,True,True,15.10,4905.95100,124.960800
6,BI-214,NaN,1155.19,True,True,1.69,4608.80700,244.069700
7,BI-214,NaN,1238.11,True,True,5.94,3257.68700,158.728500
8,BI-214,NaN,1280.96,True,True,1.47,2938.01700,202.379900
9,BI-214,NaN,1377.67,True,True,4.11,7840.99700,470.072000


### S4 - RAPPORT IDENTIFICATION AVEC CORRECTION D’INTERFERENCE

In [13]:
def extract_header_1_s4(content):
    matches = re.search(s4_header_1_pattern, content)
    if not matches:
        return None
    
    columns = [
        "Marker (X)",
        f"{matches.group(1)} {matches.group(5)}",
        "Marker (@)",
        f"{matches.group(2)} {matches.group(6)}",
        f"{matches.group(3)} {matches.group(7)} {matches.group(8)}",
        f"{matches.group(4)} {matches.group(9)}"
    ]
    
    return columns

In [14]:
def extract_header_2_s4(content):
    matches = re.search(s4_header_2_pattern, content)

    if not matches:
        return None

    columns = [
        "Marker (M/m/F)",
        f"{matches.group(1)} {matches.group(7)}",
        f"{matches.group(2)} {matches.group(8)}",
        f"{matches.group(3)} {matches.group(9)}",
        f"{matches.group(4)}",
        f"{matches.group(5)} {matches.group(10)}",
        f"{matches.group(6)} {matches.group(11)}",
    ]

    return columns

In [15]:
def extract_data_1_s4(content, header):
    matches = re.findall(s4_data_1_pattern, content)
    df = pd.DataFrame(matches, columns=header)
    
    # Change column type to bool for columns: 'Marker (X)', 'Marker (@)'
    df = df.astype({'Marker (X)': 'bool', 'Marker (@)': 'bool'})

    # Replace all instances of empty string with NaN in columns: 'Activité moyenne pondérée (mBq/g   )', 'Incert. (mBq/g   )'
    df['Activité moyenne pondérée (mBq/g   )'] = df['Activité moyenne pondérée (mBq/g   )'].replace('', np.nan)
    df['Incert. (mBq/g   )'] = df['Incert. (mBq/g   )'].replace('', np.nan)

    # Change column type to object for columns: 'Indice de confiance', 'Activité moyenne pondérée (mBq/g   )', 'Incert. (mBq/g   )'
    df = df.astype({'Indice de confiance': 'object', 'Activité moyenne pondérée (mBq/g   )': 'object', 'Incert. (mBq/g   )': 'object'})

    # Change column type to category for column: 'Nom du nucléide'
    df = df.astype({'Nom du nucléide': 'category'})
    
    return df

In [16]:
def extract_data_2_s4(content, header):
    matches = re.findall(s4_data_2_pattern, content)
    df = pd.DataFrame(matches, columns=header)
    

    # Replace all instances of empty string with NaN in columns: 'Marker (M/m/F)', 'Type du pic', 'Nucléide potentiel'
    df['Marker (M/m/F)'] = df['Marker (M/m/F)'].replace('', np.nan)
    df['Type du pic'] = df['Type du pic'].replace('', np.nan)
    df['Nucléide potentiel'] = df['Nucléide potentiel'].replace('', np.nan)

    # Change column type to category for columns: 'Marker (M/m/F)', 'Type du pic', 'Nucléide potentiel'
    df = df.astype({'Marker (M/m/F)': 'category', 'Type du pic': 'category', 'Nucléide potentiel': 'category'})

    # Change column type to float64 for columns: 'Numéro du pic', 'Energie (keV)' and 2 other columns
    df = df.astype({'Numéro du pic': 'float64', 'Energie (keV)': 'float64', 'Intensité (coups/sec)': 'float64', 'Incert.': 'float64'})
    
    return df

In [ ]:
content_s4 = sections_content[sections_titles[3]]
header_1_s4 = extract_header_1_s4(content_s4)
header_2_s4 = extract_header_2_s4(content_s4)

data_1_s4 = extract_data_1_s4(content_s4, header_1_s4)
data_2_s4 = extract_data_2_s4(content_s4, header_2_s4)

data_2_s4

,Marker (M/m/F),Numéro du pic,Energie (keV),Intensité (coups/sec),Incert.,Type du pic,Nucléide potentiel
0,NaN,1.0,16.21,0.036748,6.92,NaN,NaN
1,NaN,2.0,19.26,0.019353,12.37,NaN,NaN
2,M,3.0,25.67,0.016766,6.27,NaN,NaN
3,NaN,5.0,36.68,0.002309,88.95,NaN,NaN
4,m,8.0,53.23,0.037219,3.50,NaN,NaN
...,...,...,...,...,...,...,...
131,NaN,182.0,2879.40,0.000091,65.79,NaN,NaN
132,NaN,183.0,2921.43,0.000100,71.80,NaN,NaN
133,NaN,184.0,2939.14,0.000074,63.60,NaN,NaN
134,NaN,185.0,2977.97,0.000097,53.36,NaN,NaN


### S5 - RAPPORT LIMITES DE DETECTION

In [18]:
def extract_header_s5(content):
    matches = re.search(s5_header_pattern, content)
    
    if not matches:
        return None
    
    columns = [
        "Marker (+)",
        "Marker (?)",
        "Marker (>)",
        f"{matches.group(1)} {matches.group(8)}",
        f"{matches.group(2)} {matches.group(9)}",
        "Marker (*)",
        "Marker (@)",
        f"{matches.group(3)} {matches.group(10)}",
        f"{matches.group(4)} {matches.group(11)}",
        f"{matches.group(5)} {matches.group(12)}",
        f"{matches.group(6)} {matches.group(13)}",
        f"{matches.group(7)} {matches.group(14)}"
    ]
    
    return columns

In [19]:
def extract_data_s5(content, header):
    matches = re.findall(s5_data_pattern, content)
    df = pd.DataFrame(matches, columns=header)
    
    # Loaded variable 'data_s5' from kernel state

    # Replace all instances of empty string with NaN in column: 'Nom du nucléide'
    df['Nom du nucléide'] = df['Nom du nucléide'].replace('', np.nan)

    # Replace all instances of empty string with NaN in column: 'LD nucléide (mBq/g   )'
    df['LD nucléide (mBq/g   )'] = df['LD nucléide (mBq/g   )'].replace('', np.nan)

    # Change column type to bool for columns: 'Marker (+)', 'Marker (?)' and 3 other columns
    df = df.astype({'Marker (+)': 'bool', 'Marker (?)': 'bool', 'Marker (>)': 'bool', 'Marker (*)': 'bool', 'Marker (@)': 'bool'})

    # Change column type to float64 for columns: 'Energie (keV)', 'Intensité (%)' and 4 other columns
    df = df.astype({'Energie (keV)': 'float64', 'Intensité (%)': 'float64', 'LD Energie (mBq/g   )': 'float64', 'LD nucléide (mBq/g   )': 'float64', 'Activité (mBq/g   )': 'float64', 'SD Energie (mBq/g   )': 'float64'})

    # Replace gaps forward from the previous valid value in: 'Nom du nucléide'
    df = df.fillna({'Nom du nucléide': df['Nom du nucléide'].ffill()})
    
    return df

In [20]:
content_s5 = sections_content[sections_titles[4]]
header_s5 = extract_header_s5(content_s5)
data_s5 = extract_data_s5(content_s5, header_s5)

data_s5

,Marker (+),Marker (?),Marker (>),Nom du nucléide,Energie (keV),Marker (*),Marker (@),Intensité (%),LD Energie (mBq/g ),LD nucléide (mBq/g ),Activité (mBq/g ),SD Energie (mBq/g )
0,False,False,False,BE-7,477.59,False,False,10.42,41.850,41.850,-424.500,20.760
1,False,False,False,K-40,1460.81,False,False,10.67,32.190,32.190,-35.310,15.890
2,False,False,False,CS-137,32.19,False,False,3.66,78.760,3.495,40.120,38.770
3,False,False,False,CS-137,661.65,False,False,85.12,3.495,NaN,-19.890,1.732
4,False,False,False,TL-208,72.80,False,False,2.02,142.800,3.407,-8375.000,71.030
...,...,...,...,...,...,...,...,...,...,...,...,...
119,False,False,False,U-235,163.35,True,False,4.70,42.920,NaN,273.600,20.950
120,False,False,False,U-235,185.71,True,False,54.00,47.190,NaN,610.500,23.530
121,False,False,False,U-235,202.12,False,False,1.00,165.200,NaN,6.622,82.140
122,False,False,False,U-235,205.31,True,False,4.70,33.770,NaN,280.900,16.460


### S6 - RAPPORT LIMITES DE DETECTION  ISO 11929

In [21]:
def extract_header_s6(content):
    header = []
    matches = re.findall(s6_header_pattern, content)

    l1 = re.findall(s6_word_column_pattern, matches[0][0])
    l2 = re.findall(s6_word_column_pattern, matches[0][1])
    
    # Fusion en partant de la fin
    for a, b in zip(reversed(l1), reversed(l2)):
        header.insert(0, f"{a} {b}".strip())

    # Si l1 est plus longue, on conserve le début restant
    if len(l1) > len(l2):
        header = l1[: len(l1) - len(l2)] + header

    return header

In [22]:
def extract_data_s6(content, header):
    matches = re.findall(s6_nucleide_line_pattern, content)
    df = pd.DataFrame(matches, columns=header)
    
    # Change column type to category for column: 'Nucléide'
    df = df.astype({'Nucléide': 'category'})
    
    # Change column type to float64 for columns: 'LD', 'SD' and 6 other columns
    df = df.astype({'LD': 'float64', 'SD': 'float64', 'Limite Basse': 'float64', 'Limite Haute': 'float64', 'Moyenne Activité': 'float64', 'pondérée Incert.': 'float64', 'Meilleure Activité': 'float64', 'Estimation Incert.': 'float64'})

    return df

In [23]:
content_s6 = sections_content[sections_titles[5]]
header_s6 = extract_header_s6(content_s6)
data_s6 = extract_data_s6(content_s6, header_s6)

####### DATA ##########
data_s6

,Nucléide,LD,SD,Limite Basse,Limite Haute,Moyenne Activité,pondérée Incert.,Meilleure Activité,Estimation Incert.
0,PB-210,70.50,35.20,4880.0,5040.0,4960.0,41.60,4960.0,83.10
1,BI-214,17.20,8.55,4790.0,4910.0,4850.0,29.70,4850.0,59.50
2,PB-214,7.51,3.74,4700.0,4770.0,4730.0,18.50,4730.0,37.00
3,RA-226,214.00,107.00,5670.0,6590.0,6130.0,234.00,6130.0,467.00
4,TH-227,15.00,7.47,186.0,206.0,196.0,5.20,196.0,10.40
5,TH-230,507.00,249.00,4930.0,6790.0,5860.0,475.00,5860.0,949.00
6,PA-231,29.70,14.70,138.0,167.0,152.0,7.38,152.0,14.80
7,TH-234,64.60,28.70,3620.0,8280.0,5950.0,1190.00,5950.0,2380.00
8,U-235,9.51,4.66,231.0,245.0,238.0,3.55,238.0,7.10
9,AM-241,5.87,2.77,17.0,30.6,23.8,3.46,23.8,6.93


# Zone de Test